In [ ]:
import torch
import time
import warnings
import os
import sys
from datetime import datetime
from numba.core.errors import NumbaWarning

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from device import setup_device
from datasets.loading import get_data_loaders
from model.loading import load_model
from train.train_encoder import train_encoder
from analysis.visualization import visualize
from analysis.extraction import (
    extract_embeddings_from_model,
    extract_all_embeddings_from_model
)
from utils.checkpoint import save_checkpoint
from utils.config_io import save_configuration
from utils.cleanup import clear_gpu_cache, run_gc
from utils.printing import print_verbose
from utils.seed import set_seed
from setup.test_dir_setup import setup_testing_directory
from setup.training_setup import setup_training_environment, setup_train_config

In [ ]:
warnings.filterwarnings("ignore", message=".*force_all_finite.*")
warnings.filterwarnings("ignore", category=NumbaWarning)
warnings.filterwarnings("ignore", message=".*verbose parameter is deprecated.*")
warnings.filterwarnings("ignore", message=".*epoch parameter in `scheduler.step.*")

In [ ]:
seed = 0
clear_gpu_cache()
run_gc()
set_seed(seed)

In [ ]:
config = {
    'verbose': True,
    'seed': seed,
    'use_gpu_1_only': False,
    
    'backbone_model': 'resnet18',
    'pretrained': False,
    
    'base_dir': '../outputs/current',
    'test_group_name': 'resnet18_temperature_test',
     # test name needs to be added
    'best_model_file_name': 'best_model.pth',
    
    'data_folder': '../data/utkface/UTKFace',
    'augmentations': 'crop,flip,color,grayscale',
    'split': 'default',
    'val_size': 0.15,
    'test_size': 0.15,
    'train_size': 0.7,
    
    'batch_size': 512,
    'num_epochs': 200,
    # TODO: add optimal learning rate
    #'learning_rate': ...,
    'optimizer': 'sgd',
    # TODO: add optimal decay
    #'weight_decay': ...,
    'momentum': 0.9,
    'scheduler': 'cosine',
    'temperatures': [0.1, 0.2, 0.5, 1.0, 2.0, 5.0],
    # temperature needs to be added
        
    'key_metric': 'train_loss',
    'mode': 'min',
    'save_best_model': False,
    'use_early_stopping': False,
    'save_intermediate_models': False,
    'write_to_tensorboard': True,
    'training_metrics': [
        "embedding_norm", "embedding_variance", "knn_accuracy",
        "knr_error", "spearman", "kendall", "grad_norm", "lr"
    ],
    'nearest_neighbors': 5 # for knn or knr analysis
}

In [ ]:
def setup_subtest_environment(config, create_unique_dir=True):
    setup_testing_directory(config, create_unique_dir=create_unique_dir)
    save_configuration(config)

In [ ]:
def setup_experiment(config):
    device, model, optimizer, scheduler, monitor = setup_training_environment(config)

    train_loader, val_loader, train_loader_clean, test_loader = get_data_loaders(config)

    train_config = setup_train_config(
        device, model, optimizer, scheduler, monitor,
        train_loader, val_loader, train_loader_clean,
        config
    )
    return device, model, optimizer, scheduler, monitor, train_config

In [ ]:
# TRAINING CELL

def run_experiment(train_config, config):
    start_time = time.time()
    train_encoder(train_config, verbose=config['verbose'])
    end_time = time.time()
    print_verbose(f"Training completed in: {end_time - start_time:.2f} seconds", config['verbose'])

In [ ]:
# EXPERIMENT LOOP CELL
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
testgroup_dir_name = f"{config['test_group_name']}_{timestamp}"

for temperature in config['temperatures']:
    config['temperature'] = temperature
    t_str = f"lr{str(temperature).replace('.', 'p')}"

    subtest_name = f"{t_str}"
    subtest_dir = os.path.join(testgroup_dir_name, subtest_name)
    config['test_name'] = subtest_dir

    setup_subtest_environment(config, False)
    device, model, optimizer, scheduler, monitor, train_config = setup_experiment(config)
    run_experiment(train_config, config)